In [ ]:
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels
kill_other_ipykernels(force=True)
from nocode_robot_programming.state_decision_dataset_prepare.dataloader import TrajectoryDataset
import torch
import numpy as np

seed = 48
np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

from nocode_robot_programming.state_decision_dataset_prepare.dataset_auto import TrajectoryDatasetEvaluationViewBuilder
dataset_builder = TrajectoryDatasetEvaluationViewBuilder()
datasets = dataset_builder.load_eval_from_task("petr_kin_peg_pick")

In [ ]:
d_train, d_test, d_text = datasets[0] # hard mix
img = d_train.X[0:1]

In [ ]:
import torch
import torch.nn.functional as F

# 1) Load DINOv2
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').eval().cuda()

# 2) Find last attention
last_attn = model.blocks[-1].attn  # ViT-style block

# 3) Hook that *computes* attention from inputs using the module’s own projections
attn_maps = []  # will hold [B, H, T, T]

def capture_attn_from_inputs(module, inputs, output):
    # inputs: tuple, first is tokens x: [B, T, C]
    x = inputs[0]
    B, T, C = x.shape

    # Case A: separate q_proj/k_proj (common in DINOv2)
    if hasattr(module, 'q_proj') and hasattr(module, 'k_proj'):
        q = module.q_proj(x)  # [B, T, C]
        k = module.k_proj(x)  # [B, T, C]
        H = module.num_heads
        Dh = q.shape[-1] // H
        q = q.view(B, T, H, Dh).transpose(1, 2)  # [B, H, T, Dh]
        k = k.view(B, T, H, Dh).transpose(1, 2)  # [B, H, T, Dh]
        scale = Dh ** -0.5
        attn = (q * scale) @ k.transpose(-2, -1)  # [B, H, T, T]
        attn = attn.softmax(dim=-1)

    # Case B: fused qkv (timm-style)
    elif hasattr(module, 'qkv'):
        qkv = module.qkv(x)                    # [B, T, 3*C]
        q, k, _ = qkv.chunk(3, dim=-1)        # [B, T, C], [B, T, C]
        H = module.num_heads
        Dh = q.shape[-1] // H
        q = q.view(B, T, H, Dh).transpose(1, 2)  # [B, H, T, Dh]
        k = k.view(B, T, H, Dh).transpose(1, 2)  # [B, H, T, Dh]
        scale = Dh ** -0.5
        attn = (q * scale) @ k.transpose(-2, -1)  # [B, H, T, T]
        attn = attn.softmax(dim=-1)
    else:
        # Nothing we can do without projections
        return

    attn_maps.append(attn.detach().cpu())

h = last_attn.register_forward_hook(capture_attn_from_inputs)

# 4) Preprocess and run ONE image (self-attention is within-image; no need for batches of different images)
def prep(img_chw_0to1):
    # img_chw_0to1: [1, H, W] or [3, H, W] in [0,1]
    if img_chw_0to1.ndim == 3 and img_chw_0to1.size(0) == 1:
        img = img_chw_0to1.repeat(3, 1, 1)
    elif img_chw_0to1.ndim == 3 and img_chw_0to1.size(0) == 3:
        img = img_chw_0to1
    else:
        raise ValueError("Expect [1,H,W] or [3,H,W]")
    img = F.interpolate(img.unsqueeze(0), size=(224, 224), mode="bicubic", align_corners=False)[0]
    mean = torch.tensor([0.485, 0.456, 0.406]).cuda()[:, None, None]
    std  = torch.tensor([0.229, 0.224, 0.225]).cuda()[:, None, None]
    return ((img - mean) / std).unsqueeze(0)  # [1,3,224,224]

x = prep(img)  # demo input in [0,1]
with torch.no_grad():
    _ = model.forward_features(x)   # triggers hook

h.remove()
assert len(attn_maps) > 0, "Hook did not capture attention (check model variant)."

# 5) Convert CLS→patch attention to a 2D map
A = attn_maps[0]            # [B, H, T, T]
A = A[0]                    # [H, T, T]  (batch 0)
cls_to_all = A[:, 0, :]     # [H, T]

# Determine patch grid from model (safer than guessing)
gh, gw = (16,16) #model.patch_embed.grid_size  # e.g., (16, 16) for 224/14
num_patches = gh * gw
T = cls_to_all.shape[-1]

# Handle optional register tokens:
# tokens layout: [CLS] [registers?] [patches]
num_registers = max(0, T - 1 - num_patches)
start = 1 + num_registers
cls_to_patches = cls_to_all[:, start:start + num_patches]  # [H, num_patches]

attn_map = cls_to_patches.mean(0).reshape(gh, gw)  # avg heads -> [Gh, Gw]
attn_map = F.interpolate(attn_map[None, None], size=(x.shape[-2], x.shape[-1]),
                         mode='bilinear', align_corners=False)[0, 0]  # upsample to image size


In [ ]:
import matplotlib.pyplot as plt
plt.imshow(attn_map)

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

def _to_rgb01(img: torch.Tensor) -> torch.Tensor:
    """img: [H,W] or [1,H,W] or [3,H,W] in [0,1] -> [3,H,W] in [0,1]"""
    if img.ndim == 2:
        img = img.unsqueeze(0)                # [1,H,W]
    if img.size(0) == 1:
        img = img.repeat(3, 1, 1)             # grayscale -> rgb
    # clamp just in case
    return img.clamp(0, 1)

def _upsample_to(img_like: torch.Tensor, map_2d: torch.Tensor) -> torch.Tensor:
    """Upsample 2D map to the spatial size of img_like (3xHxW or 1xHxW or HxW)."""
    if img_like.ndim == 2:
        H, W = img_like.shape
    else:
        H, W = img_like.shape[-2:]
    if map_2d.shape[-2:] == (H, W):
        return map_2d
    m = F.interpolate(map_2d[None, None].float(), size=(H, W),
                      mode='bilinear', align_corners=False)[0, 0]
    return m

def show_img_with_attention(img: torch.Tensor,
                            attn_map: torch.Tensor,
                            alpha: float = 0.45,
                            cmap: str = 'jet',
                            title_img: str = 'Image',
                            title_overlay: str = 'Self-attention overlay'):
    """
    img: [H,W] or [1,H,W] or [3,H,W], values in [0,1]
    attn_map: [H,W] (or low-res 2D attention; will be upsampled)
    """
    # prepare tensors
    rgb = _to_rgb01(img).detach().cpu()              # [3,H,W]
    attn = attn_map.detach().cpu()
    attn = attn - attn.min()
    if attn.max() > 0:
        attn = attn / attn.max()

    # upsample attention if needed
    attn = _upsample_to(rgb, attn)

    # draw
    H, W = rgb.shape[-2:]
    fig, axs = plt.subplots(1, 2, figsize=(10, 4), dpi=120)

    # left: original
    axs[0].imshow(rgb.permute(1, 2, 0).numpy())
    axs[0].set_title(title_img)
    axs[0].axis('off')

    # right: overlay
    axs[1].imshow(rgb.permute(1, 2, 0).numpy())
    im = axs[1].imshow(attn.numpy(), cmap=cmap, alpha=alpha)
    axs[1].set_title(title_overlay)
    axs[1].axis('off')
    # optional colorbar
    cbar = fig.colorbar(im, ax=axs[1], fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel('attention', rotation=270, labelpad=12)

    plt.tight_layout()
    plt.show()


In [ ]:
show_img_with_attention(img, attn_map)